# Advanced Retry Decorator with Exponential Backoff and Jitter

## Exercise: Advanced Retry Decorator with Custom Exception

You are a DevOps engineer writing scripts that interact with a critical but occasionally flaky API. When the API has a momentary outage, all client scripts retry at the exact same time, causing a "thundering herd" that overloads the API as soon as it comes back online.

Your task is to implement a robust, production-ready retry decorator that uses exponential backoff, random jitter, and provides clear, contextual errors upon final failure.

## Functional Requirements:

- The factory must accept three arguments: `max_attempts`, `base_delay`, and `jitter`.
- If the wrapped function call succeeds, its return value should be returned immediately.
- If the call fails by raising an `Exception`, the decorator must retry the function up to `max_attempts`.
- The delay between retries must use exponential backoff: `delay = base_delay * (2 ** (failure_count - 1))`.
- To prevent synchronized retries, a random jitter (`random.uniform(0, jitter)`) must be added to each delay.
- The decorator must preserve the original function's metadata.

## Error Handling and Custom Exceptions:

### Factory Input Validation:

The `retry_with_backoff` factory must validate its own arguments when it is called.

- `max_attempts` must be an integer greater than `0`.
- `base_delay` and `jitter` must be non-negative numbers.
- If validation fails, it must raise a `ValueError`.

### Custom Exception:

You must define a new custom exception named `MaxRetriesExceededError` that inherits from `Exception`.

- Its `__init__` method should accept the total attempts made and the `last_exception` that occurred.
- It should store these as attributes and generate a clear error message.

### Final Failure:

If the function is still failing on its final attempt, the decorator must raise your new `MaxRetriesExceededError`.

### Exception Chaining:

To preserve the full context of the original failure, the `MaxRetriesExceededError` must be raised from the last exception caught (using `raise ... from ...`).

## Example:

```python
class MaxRetriesExceededError(Exception):
    # ... implementation ...
    pass

@retry_with_backoff(max_attempts=3, base_delay=0.1, jitter=0.05)
def connect_to_api():
    # ... code that might raise ConnectionError ...
    pass

try:
    connect_to_api()
except MaxRetriesExceededError as e:
    print(f"Failed after {e.attempts} attempts. Last error was: {e.last_exception}")
```

## How Your Solution Will Be Tested:

- A function that succeeds on the first try.
- A function that fails a few times before succeeding.
- The factory's own argument validation.
- A function that fails all attempts, which must raise `MaxRetriesExceededError`.
- Verification that the raised `MaxRetriesExceededError` contains the correct number of attempts and that the original, underlying exception is chained correctly (`__cause__`).

In [ ]:
import functools
import random
import time
from email import message


class MaxRetriesExceededError(Exception):
    def __init__(self, attempts, last_exception):
        self.attempts = attempts
        self.last_exception = last_exception

        message = (
            f"Failed after {attempts} attempts. "
            f"Last error: {last_exception}"
        )
        super().__init__(message)


def retry_with_backoff(max_attempts,
                       base_delay=1.0,
                       jitter=0.1):

    # Factory validation
    if (
        not isinstance(max_attempts, int)
        or max_attempts <= 0
    ):
        raise ValueError(
            "max_attempts must be a positive integer."
        )

    if (
        not isinstance(base_delay, (int, float))
        or base_delay < 0
    ):
        raise ValueError(
            "base_delay must be a non-negative number."
        )

    if (
        not isinstance(jitter, (int, float))
        or jitter < 0
    ):
        raise ValueError(
            "jitter must be a non-negative number."
        )

    def decorator(func):

        @functools.wraps(func)
        def wrapper(*args, **kwargs):

            last_exception = None

            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)

                except Exception as e:
                    last_exception = e

                    if attempt == max_attempts:
                        raise MaxRetriesExceededError(
                            attempt,
                            last_exception
                        ) from last_exception

                    delay = (
                        base_delay
                        * (2 ** (attempt - 1))
                    )

                    delay += random.uniform(
                        0,
                        jitter
                    )

                    time.sleep(delay)
            return None

        return wrapper

    return decorator

In [ ]:
import functools
import random
import time

class MaxRetriesError:
    def __init__(self , attempts, last_exception):
        self.attempts = attempts
        self.last_exception = last_exception

        messages = (
            f"Failed after {attempts} attempts. "
            f"Last error: {last_exception}"
        )
        super().__init__(messages)

def retry_with_backoff(max_attempts,
                       base_delay=1.0,
                       jitter=0.1):
    if not isinstance(max_attempts, int) or max_attempts <= 0:
        raise ValueError("max_attempts must be a positive integer.")
    if not isinstance(base_delay , (int, float)) or base_delay < 0:
        raise ValueError("base_delay must be a non-negative number.")
    if not isinstance(jitter , (int, float)) or jitter < 0:
        raise ValueError("jitter must be a non-negative number.")

    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exception = e
                    if attempt == max_attempts:
                        raise MaxRetriesExceededError(attempt, last_exception) from last_exception
                    delay = (base_delay * (2 ** (attempt - 1)))
                    delay += random.uniform(0, jitter)
                    time.sleep(delay)

                return None
            return wrapper
        return decorator
